# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** The required ORNL DAAC CSV could not be downloaded anonymously (HTTPS 401 / S3 AccessDenied); NASA Earthdata Login is required on the tested routes. No manual login validation was performed. Acquisition failed and downstream analysis is unverified.

Attempt status: blocked. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Ground far-red sun-induced chlorophyll fluorescence and vegetation indices in the US Midwestern agroecosystems.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-24b915f099a92f385eba9e54c3b86e2f)


# Ground far-red SIF and vegetation indices in the US Midwest — dataset validation walkthrough

Reproducing the **Technical Validation** of: Wu G., Guan K., Kimm H., Miao G., Yang X., Jiang C. (2024). *Ground far-red sun-induced chlorophyll fluorescence and vegetation indices in the US Midwestern agroecosystems.* **Scientific Data 11**, 228. https://doi.org/10.1038/s41597-024-03004-w

Data: ORNL DAAC DOI [10.3334/ORNLDAAC/2136](https://doi.org/10.3334/ORNLDAAC/2136) — one public CSV of half-hourly far-red SIF (five retrieval methods), calibration factors, PAR/FPAR, and six vegetation indices from FluoSpec2 systems at seven US Corn Belt sites (2016–2021).

**Scope:** a single-example, CPU-only validation of the *published* dataset. We recompute the paper's cross-method SIF agreement, SIF–VI relationships, a seasonal time series, and an LMG relative-importance decomposition. We do **not** re-run the raw-spectrum MATLAB retrieval pipeline (raw spectra are not published); the author's MATLAB functions ([wugh16/SIF_VI_process_functions](https://github.com/wugh16/SIF_VI_process_functions), commit `4b4c572b`) were inspected to confirm the VI formulas, which match the dataset's variable table.

**注意（日本語）:** このノートブックは、2024年 Scientific Data 論文で公開された地上SIF・植生指数データセット（ORNL DAAC, 12 MB CSV）を読み込み、論文の「Technical Validation」で行われた解析（5手法間のクロス比較、SIF–VI関係、季節変化、LMG分解）を最小構成で再現します。生スペクトルからのMATLAB処理パイプラインの再実行は対象外です。所要時間は10分以内、ダウンロードは約12 MBです。

## Runtime selection

**Select Runtime → Change runtime type → CPU (standard).** No GPU is needed: the analysis is tabular (37.5k rows × 32 columns) using pandas/numpy/scipy/matplotlib.

**ランタイム（日本語）:** 「ランタイム → ランタイムのタイプを変更 → CPU」を選択してください。GPUは不要で、表形式の解析のみ行います。

## Setup

Only standard preinstalled Colab packages are used. The cell below prints their versions so the environment is recorded in the saved output.

In [ ]:
import sys, subprocess, platform
# boto3 is not preinstalled on the Colab runtime; install it quietly (verified needed 2026-10-01)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3"], check=True, timeout=600)
import numpy, pandas, scipy, matplotlib

print("python", sys.version.split()[0], "|", platform.machine())
print("numpy", numpy.__version__, "| pandas", pandas.__version__)
print("scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

## Data acquisition with provenance

The dataset is a single public CSV (~12 MB, 37,501 rows × 32 columns) plus a published `.sha256` sidecar. We try the documented ORNL DAAC public URLs in order, verify the HTTP status / final URL, download **in Colab**, and check the SHA-256 against the author repository's published checksum. **データ取得（日本語）:** ORNL DAACの公開URLからCSVをColab内でダウンロードし、公開されているSHA-256と照合します。

In [ ]:
import hashlib, os, re, urllib.request

CSV_NAME = "SIF_VegIndices_Illinois_Nebraska_Halfhour.csv"
ATTEMPTS = [
    "https://daac.ornl.gov/daacdata/cms/Crops_SIF_VegIndices_IL_NE/data/" + CSV_NAME,
    "https://data.ornldaac.earthdata.nasa.gov/public/cms/Crops_SIF_VegIndices_IL_NE/data/" + CSV_NAME,
]
SHA_URL = "https://data.ornldaac.earthdata.nasa.gov/public/cms/Crops_SIF_VegIndices_IL_NE/data/" + CSV_NAME + ".sha256"

UA = {"User-Agent": "phenopaper-colab-validation/1.0"}

def http_get(url, timeout=180):
    req = urllib.request.Request(url, headers=UA)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.status, r.geturl(), r.headers.get("Content-Type", ""), r.read()

raw = None
for url in ATTEMPTS:
    try:
        status, final_url, ctype, payload = http_get(url)
        print(f"GET {status} {len(payload)/1e6:.2f} MB  ctype={ctype}\n  final={final_url}")
        if status == 200 and payload[:1] != b"<" and len(payload) > 5_000_000:
            raw = payload
            print("selected:", url)
            break
    except Exception as e:
        print("failed:", url, "->", type(e).__name__, e)

# Fallback: CMR DirectDistributionInformation lists the anonymous-public S3 bucket
# ornl-cumulus-prod-public (region us-west-2) for this open-access dataset.
if raw is None:
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    s3 = boto3.client("s3", region_name="us-west-2",
                      config=Config(signature_version=UNSIGNED))
    bucket = "ornl-cumulus-prod-public"
    prefix = "cms/Crops_SIF_VegIndices_IL_NE/"
    keys = []
    try:
        paginator = s3.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            keys += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(CSV_NAME)]
        print("anonymous S3 listing found:", keys)
    except Exception as e:
        print("anonymous S3 listing failed:", type(e).__name__, e)
        keys = [prefix + "data/" + CSV_NAME]
    for key in keys:
        try:
            obj = s3.get_object(Bucket=bucket, Key=key)
            payload = obj["Body"].read()
            print(f"S3 GET s3://{bucket}/{key}: {len(payload)/1e6:.2f} MB")
            if len(payload) > 5_000_000:
                raw = payload
                print("selected: anonymous S3", bucket, key)
                break
        except Exception as e:
            print("S3 GET failed:", key, "->", type(e).__name__, e)

assert raw is not None, "no public route returned the CSV; report exact failures"

sha_remote = None
try:
    s, _, _, body = http_get(SHA_URL)
    m = re.search(rb"([0-9a-f]{64})", body)
    if m:
        sha_remote = m.group(1).decode()
        print("published sha256:", sha_remote)
except Exception as e:
    print("sha sidecar unavailable:", type(e).__name__, e)

sha_local = hashlib.sha256(raw).hexdigest()
print("downloaded sha256:", sha_local)
if sha_remote:
    assert sha_local == sha_remote, "checksum mismatch against published sidecar"
    print("SHA-256 matches the published checksum.")
else:
    print("WARNING: sidecar checksum not available; size/signature checks only.")

os.makedirs("/content/data", exist_ok=True)
with open("/content/data/" + CSV_NAME, "wb") as f:
    f.write(raw)
print("saved /content/data/" + CSV_NAME)

## Load the dataset and apply the documented fill value

Per the ORNL DAAC user guide, missing measurements are coded `-9999`; we map them to NaN. Each row is one half hour at one of 7 sites (Nebraska: US-Ne2/US-Ne3; Illinois: US-UiB/US-UiC/Reifsteck/Rund/Reinhart), 2016–2021. **データ読み込み（日本語）:** 欠測値 `-9999` をNaNに変換し、サイト・年・作物の構成を確認します。

In [ ]:
df = pd.read_csv("/content/data/" + CSV_NAME)
print("shape:", df.shape)
print("columns:\n", list(df.columns))

df = df.replace(-9999, float("nan"))
print("\nsite/year/species coverage:")
print(df.groupby(["site", "year", "species"]).size().rename("rows").to_string())

## Analysis 1 — Cross-method agreement of the five SIF760 retrievals

The paper compared sFLD, 3FLD, iFLD, SFM-linear and SFM-nonlinear SIF760 and reports **R² > 0.93 between methods when the enclosure was temperature-controlled (≤ 25 °C)**; SFM retrievals degrade at higher enclosure temperature. We compute pairwise R² within each enclosure-temperature regime. **解析1（日本語）:** 5つのSIF760取得手法間の決定係数R²を、エンクロージャ温度25 ℃以下／超過に分けて計算します。

In [ ]:
SIF_METHODS = ["SIF_sFLD_raw", "SIF_3FLD_raw", "SIF_iFLD_raw",
               "SIF_SFM_nonlinear_raw", "SFM_linear_raw"]

min_rows = 30  # require enough paired half-hours for a meaningful R²

def pairwise_r2(sub):
    rows = []
    for i, a in enumerate(SIF_METHODS):
        for b in SIF_METHODS[i+1:]:
            pair = sub[[a, b]].dropna()
            if len(pair) < min_rows:
                continue
            r = np.corrcoef(pair[a], pair[b])[0, 1]
            rows.append({"method_a": a, "method_b": b,
                         "n": len(pair), "r2": round(r**2, 3)})
    return pd.DataFrame(rows)

cool = df[df["enclosure_temp"] <= 25]
warm = df[df["enclosure_temp"] > 25]
r2_cool = pairwise_r2(cool)
r2_warm = pairwise_r2(warm)

print("=== enclosure_temp <= 25 C (controlled; paper: R2 > 0.93) ===")
print(r2_cool.to_string(index=False))
print("\n=== enclosure_temp > 25 C ===")
print(r2_warm.to_string(index=False))

A scatter of one method pair makes the regime difference visible. The grey diagonal is 1:1.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, (label, sub) in zip(axes, [("enclosure <= 25 C", cool), ("enclosure > 25 C", warm)]):
    pair = sub[["SIF_3FLD_raw", "SFM_linear_raw"]].dropna()
    ax.scatter(pair["SIF_3FLD_raw"], pair["SFM_linear_raw"], s=3, alpha=0.25)
    lim = [0, max(2.0, np.nanpercentile(pair.values, 99))]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("SIF_3FLD_raw (mW m$^{-2}$ nm$^{-1}$ sr$^{-1}$)")
    ax.set_title(f"3FLD vs SFM-linear — {label} (n={len(pair)})")
axes[0].set_ylabel("SIF760 (mW m$^{-2}$ nm$^{-1}$ sr$^{-1}$)")
fig.suptitle("Cross-method SIF760 agreement by enclosure temperature")
plt.tight_layout(); plt.show()

## Analysis 2 — SIF760 vs vegetation indices, with and without PAR

The paper relates half-hourly SIF760 to each VI (R² 0.20–0.40) and to **VI × PAR** (R² 0.56–0.88). We use 3FLD SIF760 multiplied by the published calibration-adjustment factor `f_cal_corr_QEPRO` (which corrects calibrating-light-source degradation), and report R² for each VI and each VI×PAR regressor. **解析2（日本語）:** 補正済みSIF760（3FLD×校正係数）と各植生指数、および指数×PARとのR²を、論文のレンジ（0.20–0.40／0.56–0.88）と比較します。

In [ ]:
VIS = ["NDVI", "EVI", "NIRv", "CI_red_edge", "CI_green", "PRI"]
sif_corr = df["SIF_3FLD_raw"] * df["f_cal_corr_QEPRO"]

def r2(y, x):
    m = np.isfinite(y) & np.isfinite(x)
    if m.sum() < min_rows:
        return np.nan, int(m.sum())
    return float(np.corrcoef(y[m], x[m])[0, 1] ** 2), int(m.sum())

rows = []
for vi in VIS:
    r_vi, n_vi = r2(sif_corr.values, df[vi].values)
    r_par, n_par = r2(sif_corr.values, (df[vi] * df["PAR"]).values)
    rows.append({"VI": vi, "R2_SIF_vs_VI": round(r_vi, 3), "n": n_vi,
                 "R2_SIF_vs_VIxPAR": round(r_par, 3), "n_par": n_par})
vi_table = pd.DataFrame(rows)
print("paper reference: R2(SIF~VI) 0.20-0.40 ; R2(SIF~VIxPAR) 0.56-0.88 (half-hourly)")
print(vi_table.to_string(index=False))

## Analysis 3 — Seasonal dynamics for one site-year (US-Ne2, 2016, corn)

A deterministic example: the irrigated US-Ne2 site, 2016 corn season. Daily means of calibration-corrected 3FLD SIF760, NIRv, NDVI and PAR show the expected bell-shaped growing-season trajectory. **解析3（日本語）:** US-Ne2サイト2016年トウモロコシの季節変化を日平均でプロットします。

In [ ]:
sy = df[(df["site"] == "US-Ne2") & (df["year"] == 2016)].copy()
sy["timestamp_start"] = pd.to_datetime(sy["timestamp_start"])
sy["sif_corr"] = sy["SIF_3FLD_raw"] * sy["f_cal_corr_QEPRO"]
daily = sy.set_index("timestamp_start")[["sif_corr", "NIRv", "NDVI", "PAR"]].resample("D").mean()

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(daily.index, daily["sif_corr"], color="tab:red", label="SIF760 (3FLD, corrected)")
ax1.set_ylabel("SIF760 (mW m$^{-2}$ nm$^{-1}$ sr$^{-1}$)", color="tab:red")
ax1.set_xlabel("date (local time, US Central)")
ax2 = ax1.twinx()
ax2.plot(daily.index, daily["NIRv"], color="tab:green", label="NIRv")
ax2.plot(daily.index, daily["NDVI"], color="tab:blue", label="NDVI")
ax2.plot(daily.index, daily["PAR"] / daily["PAR"].max(), color="gray", ls=":", label="PAR (normalized)")
ax2.set_ylabel("index / PAR (normalized)")
lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lab1 + lab2, loc="upper left", fontsize=9)
ax1.set_title("US-Ne2, 2016 corn — daily means (notebook-created visualization)")
plt.tight_layout(); plt.show()
print(f"days with data: {daily['sif_corr'].notna().sum()}, "
      f"peak daily SIF760 = {daily['sif_corr'].max():.3f} mW m-2 nm-1 sr-1")

## Analysis 4 — LMG relative-importance decomposition (method demonstration)

The paper decomposes peak-season SIF760 variation with the LMG (Lindeman–Merenda–Gold) relative-importance method using fPAR, PAR, ΦF, canopy structure and fesc. The published CSV does not contain derived ΦF/fesc columns, so this cell demonstrates the **LMG method itself** on the available regressors `[PAR, FPAR_VI, NIRv]` for peak-season half-hours (DOY 160–240) at US-Ne2/2016. LMG = the average R² gain of each regressor over all orderings; we implement it directly with numpy. This is **not** the paper's exact decomposition. **解析4（日本語）:** 論文と同じLMG相対的重要度法を、CSVに含まれる PAR・FPAR_VI・NIRv の3回帰子で実演します（論文そのものの分解ではありません）。

In [ ]:
from itertools import permutations

peak = sy[(sy["doy"] >= 160) & (sy["doy"] <= 240)]
regressors = ["PAR", "FPAR_VI", "NIRv"]
sub = peak[["sif_corr"] + regressors].dropna()
print("peak-season half-hours used:", len(sub))

Y = sub["sif_corr"].values
X = {r: sub[r].values for r in regressors}

def r2_of(cols):
    if not cols:
        return 0.0
    Xm = np.column_stack([X[c] for c in cols])
    Xm = np.column_stack([np.ones(len(Y)), Xm])
    beta, *_ = np.linalg.lstsq(Xm, Y, rcond=None)
    pred = Xm @ beta
    ss_res = ((Y - pred) ** 2).sum()
    ss_tot = ((Y - Y.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot

lmg = {r: 0.0 for r in regressors}
for perm in permutations(regressors):
    prev = 0.0
    for k in range(1, len(perm) + 1):
        cur = r2_of(list(perm[:k]))
        lmg[perm[k-1]] += cur - prev
        prev = cur
n_ord = len(list(permutations(regressors)))
lmg = {r: v / n_ord for r, v in lmg.items()}
total_r2 = r2_of(regressors)

lmg_df = pd.DataFrame({"regressor": list(lmg),
                       "LMG_share": [round(lmg[r] / total_r2, 3) for r in lmg]}).sort_values("LMG_share", ascending=False)
print(f"full-model R2 = {total_r2:.3f}")
print("LMG share of explained variance (sums to 1):")
print(lmg_df.to_string(index=False))

plt.figure(figsize=(5, 3))
plt.bar(lmg_df["regressor"], lmg_df["LMG_share"], color="tab:purple")
plt.ylabel("LMG share of R2")
plt.title("US-Ne2 2016 peak season — LMG decomposition (notebook demonstration)")
plt.tight_layout(); plt.show()

## Interpretation and validation record

- **Cross-method agreement** should show R² well above 0.9 between FLD-family methods under controlled enclosure temperature, with SFM-based pairs degrading when `enclosure_temp` exceeds 25 °C (paper's reported caveat).
- **SIF–VI relationships**: half-hourly R² against VIs alone is expected in the 0.2–0.4 range, rising substantially when PAR is included (VI × PAR), consistent with APAR dominating canopy SIF variation.
- **Seasonal dynamics**: SIF760 and structural VIs (NIRv/NDVI) peak together mid-season.
- **LMG demonstration**: illustrates how SIF variance is partitioned among PAR, canopy absorbed-PAR proxy (FPAR_VI) and structure (NIRv); the paper's own decomposition additionally used derived ΦF/fesc terms not present in the CSV.

**検証結果の見方（日本語）:** 手法間R²が25 ℃以下で0.9超になること、SIF–VIのR²がPAR導入で大きく上がることが論文の記載と整合するかを、上の表で確認してください。単一データセットの再計算であり、論文の全結果の検証ではありません。

**Sources:** Wu et al. 2024 (Sci Data 11:228); ORNL DAAC 10.3334/ORNLDAAC/2136 (CSV + user guide); author MATLAB code `wugh16/SIF_VI_process_functions@4b4c572b` (MIT). LMG method: Lindeman, Merenda & Gold (1980). The scatter, time-series and bar plots are notebook-created visualizations, not author figures.